# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [2]:
findings = {
    "Finding #1: Anatomy of Growing Content": {
        "label_source": "trend_direction (30d vs prior-30d impressions, rule-based)",
        "validation_design": "cross-sectional comparison, observational — not causal",
        "n": "74,187 up vs 45,272 down",
    },
    "Finding #2: What Predicts Health (RF feature importance)": {
        "label_source": "health_score = impressions(30) + position(30) + ctr(20) + scroll(20)",
        "validation_design": "holdout-tested, BUT top features are formula components -> label leakage",
        "n": "61,790 active content records",
    },
}
for name, info in findings.items():
    print(name)
    for k, v in info.items():
        print(f"  {k}: {v}")
    print()

Finding #1: Anatomy of Growing Content
  label_source: trend_direction (30d vs prior-30d impressions, rule-based)
  validation_design: cross-sectional comparison, observational — not causal
  n: 74,187 up vs 45,272 down

Finding #2: What Predicts Health (RF feature importance)
  label_source: health_score = impressions(30) + position(30) + ctr(20) + scroll(20)
  validation_design: holdout-tested, BUT top features are formula components -> label leakage
  n: 61,790 active content records



## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
!git clone https://github.com/MarriamFatima-alt/flyrank-ml-internship.git
%cd flyrank-ml-internship
!python scripts/01_prepare_features.py
!python scripts/02_baseline_score.py
!python scripts/03_train_model.py

import sys, json
sys.path.insert(0, "scripts")
import pandas as pd, numpy as np
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, precision_at_k
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

frame = pd.read_csv("data/processed/refresh_feature_vector.csv")

def build_feature_matrix(frame):
    numeric = [c for c in MODEL_NUMERIC_FEATURES if c in frame.columns]
    categorical = [c for c in MODEL_CATEGORICAL_FEATURES if c in frame.columns]
    num = frame[numeric].apply(pd.to_numeric, errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0)
    cat = frame[categorical].fillna("unknown").astype(str)
    enc = pd.get_dummies(cat, prefix=categorical, dummy_na=False, dtype=float)
    return pd.concat([num.reset_index(drop=True), enc.reset_index(drop=True)], axis=1)

X = build_feature_matrix(frame)
y = frame["is_declining_label"].astype(int)

# BEFORE: naive random row split (ignores that rows can share a client)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=42)
rf.fit(Xtr, ytr)
proba_before = rf.predict_proba(Xte)[:, 1]

clients_train = set(frame.iloc[Xtr.index]["client_id"])
clients_test = set(frame.iloc[Xte.index]["client_id"])
leaking_clients = len(clients_train & clients_test)

print("BEFORE — naive random row split:")
print(f"  ROC AUC:        {roc_auc_score(yte, proba_before):.3f}")
print(f"  Precision@50:   {precision_at_k(yte, proba_before, 50):.3f}")
print(f"  Avg precision:  {average_precision_score(yte, proba_before):.3f}")
print(f"  Clients appearing in BOTH train and test: {leaking_clients} of {frame['client_id'].nunique()}")

# AFTER: official client-holdout split (already used in Week-5 model)
with open("outputs/model_results.json") as f:
    after = json.load(f)["models"]["random_forest"]

print("\nAFTER — client-holdout split (Week-5 model, honest):")
print(f"  ROC AUC:        {after['roc_auc']:.3f}")
print(f"  Precision@50:   {after['precision_at_50']:.3f}")
print(f"  Avg precision:  {after['average_precision']:.3f}")

comparison = pd.DataFrame({
    "ROC AUC": [round(roc_auc_score(yte, proba_before), 3), round(after["roc_auc"], 3)],
    "Precision@50": [round(precision_at_k(yte, proba_before, 50), 3), round(after["precision_at_50"], 3)],
    "Avg Precision": [round(average_precision_score(yte, proba_before), 3), round(after["average_precision"], 3)],
}, index=["Before (random split)", "After (client-holdout)"])
comparison

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 128 (delta 43), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 1.85 MiB | 3.74 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/flyrank-ml-internship
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv
Wrote baseline queue: /content/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship/outputs/model_results.json
BEFORE — naive random row split:
  ROC AUC:        0.758
  Prec

,ROC AUC,Precision@50,Avg Precision
Before (random split),0.758,0.90,0.768
After (client-holdout),0.750,0.74,0.618


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [11]:
overlap = frame[["impressions_90d", "impressions_last_30d", "impressions_prev_30d"]].copy()
overlap["last60_sum"] = overlap["impressions_last_30d"] + overlap["impressions_prev_30d"]
corr = overlap["impressions_90d"].corr(overlap["last60_sum"])
coverage = (overlap["last60_sum"] / overlap["impressions_90d"].replace(0, np.nan)).mean()
print(f"Correlation between impressions_90d and (last30d+prev30d): {corr:.2f}")
print(f"Avg share of the 90d window that overlaps the label's 60d window: {coverage:.1%}")

def run_split(features, label):
    num = frame[[c for c in features if c in frame.columns]].apply(pd.to_numeric, errors="coerce").fillna(0)
    cat = frame[MODEL_CATEGORICAL_FEATURES].fillna("unknown").astype(str)
    enc = pd.get_dummies(cat, prefix=MODEL_CATEGORICAL_FEATURES, dummy_na=False, dtype=float)
    Xf = pd.concat([num.reset_index(drop=True), enc.reset_index(drop=True)], axis=1)

    # Filter by client_id to create train and test sets
    Xtr2 = Xf[~frame["client_id"].isin(clients_test)]
    Xte2 = Xf[frame["client_id"].isin(clients_test)]
    ytr2 = y[~frame["client_id"].isin(clients_test)]
    yte2 = y[frame["client_id"].isin(clients_test)]

    # Check if the training set contains both classes
    if ytr2.nunique() < 2:
        print(f"Warning for '{label}': Training set for client-holdout split contains only {ytr2.nunique()} unique class(es). Cannot train a binary classifier meaningfully.")
        print(f"Skipping evaluation for '{label}'.")
        return

    # Check if the test set is empty or monochromatic for evaluation metrics
    if yte2.empty or yte2.nunique() < 2:
        print(f"Warning for '{label}': Test set for client-holdout split is empty or contains only {yte2.nunique()} unique class(es). Cannot evaluate binary classification metrics meaningfully.")
        print(f"Skipping evaluation for '{label}'.")
        return

    rf2 = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=42)
    rf2.fit(Xtr2, ytr2)

    proba_output = rf2.predict_proba(Xte2)

    # If predict_proba unexpectedly returns fewer than 2 columns, it means we can't get P(class=1).
    if proba_output.shape[1] < 2:
        print(f"Error for '{label}': predict_proba returned {proba_output.shape[1]} column(s) unexpectedly. Expected 2 for binary classification. Cannot compute P(class=1).")
        print(f"Skipping evaluation for '{label}'.")
        return

    p2 = proba_output[:, 1]
    print(f"{label}: ROC AUC={roc_auc_score(yte2, p2):.3f}  Precision@50={precision_at_k(yte2, p2, 50):.3f}")

run_split(MODEL_NUMERIC_FEATURES, "WITH log_impressions_90d")
run_split([f for f in MODEL_NUMERIC_FEATURES if f != "log_impressions_90d"], "WITHOUT log_impressions_90d")

flag_like = [c for c in MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES if "flag" in c.lower() or "health" in c.lower()]
print(f"\nProduct-flag / composite-score columns used as features: {flag_like if flag_like else 'None found'}")

print(f"\nBase rate (declining rows): {y.mean():.1%}")

pred = pd.read_csv("data/processed/model_predictions.csv")
test = pred[pred["split"] == "test"].copy()
test["pred_label"] = (test["best_model_probability"] >= 0.5).astype(int)
fp = test[(test["is_declining_label"] == 0) & (test["pred_label"] == 1)].sort_values("best_model_probability", ascending=False)
fn = test[(test["is_declining_label"] == 1) & (test["pred_label"] == 0)].sort_values("best_model_probability")
print(f"\nFalse positives: {len(fp)}  |  False negatives: {len(fn)}")
print("\nExample false positive (predicted declining, actually growing):")
print(fp[["content_id", "client_id", "best_model_probability"]].head(1))
print("\nExample false negative (actually declining, model missed it):")
print(fn[["content_id", "client_id", "best_model_probability"]].head(1))


Correlation between impressions_90d and (last30d+prev30d): 0.98
Avg share of the 90d window that overlaps the label's 60d window: 56.2%
Warning for 'WITH log_impressions_90d': Training set for client-holdout split contains only 1 unique class(es). Cannot train a binary classifier meaningfully.
Skipping evaluation for 'WITH log_impressions_90d'.
Warning for 'WITHOUT log_impressions_90d': Training set for client-holdout split contains only 1 unique class(es). Cannot train a binary classifier meaningfully.
Skipping evaluation for 'WITHOUT log_impressions_90d'.

Product-flag / composite-score columns used as features: None found

Base rate (declining rows): 54.2%

False positives: 236  |  False negatives: 296

Example false positive (predicted declining, actually growing):
                 content_id          client_id  best_model_probability
25913  content_331182ca4cae  client_f74efabef1                 0.74612

Example false negative (actually declining, model missed it):
               

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [12]:
original_claim = "Random Forest catches 74% of declining pages in the top 50 — this model finds the content that's about to lose traffic."

rewritten_claim = (
    "On a client-holdout test set, the model's top-50 ranked pages contained a declining "
    "page 74% of the time, vs a 54.2% base rate in the data (observed, measured). This is "
    "decision-support for review prioritization, not a causal or guaranteed prediction for "
    "any single page (directional, not causal)."
)

print("BEFORE:", original_claim)
print()
print("AFTER: ", rewritten_claim)
print()
print(f"Base rate check: {y.mean():.1%} (declining rows in full dataset)")

BEFORE: Random Forest catches 74% of declining pages in the top 50 — this model finds the content that's about to lose traffic.

AFTER:  On a client-holdout test set, the model's top-50 ranked pages contained a declining page 74% of the time, vs a 54.2% base rate in the data (observed, measured). This is decision-support for review prioritization, not a causal or guaranteed prediction for any single page (directional, not causal).

Base rate check: 54.2% (declining rows in full dataset)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.